# 03. 생활인구 데이터 클리닝

**처리 대상**
- `raw/living_population/LOCAL_PEOPLE_GU_{year}.zip` (2020~2024) — 내국인 생활인구
- `raw/living_population/LONG_FOREIGNER_GU_{year}.zip` (2020~2024) — 장기체류 외국인 생활인구

**구조**
- 기준일ID(일별) × 시간대구분(24구간) × 자치구코드 → 총생활인구수
- 2024년부터 컬럼명이 영문으로 변경됨 (자동 처리)

**처리 로직**
1. 일별 시간대 합산 → 일 생활인구
2. 월별 일 평균 생활인구
3. 연간 월 평균 → 연 평균 일일 생활인구

**출력**
- `processed/living_population/living_pop_daily_avg_by_gu_year.csv`
- `processed/living_population/foreigner_lp_daily_avg_by_gu_year.csv`
- `processed/living_population/living_pop_merged_by_gu_year.csv`

In [ ]:
import pandas as pd
import numpy as np
import zipfile
import io
from pathlib import Path

BASE = Path("../../data")
RAW  = BASE / "raw" / "living_population"
OUT  = BASE / "processed" / "living_population"
OUT.mkdir(parents=True, exist_ok=True)

GU_CODE = {
    11110:"종로구", 11140:"중구",     11170:"용산구",   11200:"성동구",   11215:"광진구",
    11230:"동대문구",11260:"중랑구",  11290:"성북구",   11305:"강북구",   11320:"도봉구",
    11350:"노원구",  11380:"은평구",  11410:"서대문구", 11440:"마포구",   11470:"양천구",
    11500:"강서구",  11530:"구로구",  11545:"금천구",   11560:"영등포구", 11590:"동작구",
    11620:"관악구",  11650:"서초구",  11680:"강남구",   11710:"송파구",   11740:"강동구"
}

# 2024년 영문 컬럼명 → 한글 매핑
EN_TO_KR = {
    "stdr_de_id":    "기준일ID",
    "tmzon_pd_se":   "시간대구분",
    "adstrd_code_se":"자치구코드",
    "tot_lvpop_co":  "총생활인구수"
}
print("설정 완료")

In [ ]:
def read_lp_zip(zip_path: Path) -> pd.DataFrame:
    """생활인구 zip 파일을 읽어 자치구×일×시간대 데이터프레임 반환"""
    with zipfile.ZipFile(zip_path, "r") as z:
        fname = [n for n in z.namelist() if n.endswith(".csv")][0]
        with z.open(fname) as f:
            raw = f.read()
    df = pd.read_csv(io.BytesIO(raw), encoding="euc-kr", low_memory=False)
    df.columns = df.columns.str.strip()
    df = df.rename(columns=EN_TO_KR)
    df["자치구코드"] = pd.to_numeric(df["자치구코드"], errors="coerce")
    df = df[df["자치구코드"].isin(GU_CODE.keys())].copy()
    df["gu"] = df["자치구코드"].map(GU_CODE)
    df["date"] = pd.to_datetime(df["기준일ID"].astype(str), format="%Y%m%d")
    df["year"]  = df["date"].dt.year
    df["month"] = df["date"].dt.month
    return df

def aggregate_to_annual(df: pd.DataFrame, pop_col: str) -> pd.DataFrame:
    """시간대별 → 일합산 → 월평균 → 연평균"""
    df_day = (
        df.groupby(["year","month","date","gu"])[pop_col]
        .sum().reset_index()
    )
    df_month = (
        df_day.groupby(["year","month","gu"])[pop_col]
        .mean().reset_index()
    )
    df_year = (
        df_month.groupby(["year","gu"])[pop_col]
        .mean().reset_index()
    )
    return df_year

print("함수 정의 완료")

## 1. 내국인 생활인구 (연 평균 일일 생활인구)

In [ ]:
records = []
for year in range(2020, 2025):
    zip_path = RAW / f"LOCAL_PEOPLE_GU_{year}.zip"
    if not zip_path.exists():
        print(f"  {year}: 파일 없음")
        continue
    df = read_lp_zip(zip_path)
    pop_col = "총생활인구수"
    df_y = aggregate_to_annual(df, pop_col)
    df_y = df_y.rename(columns={pop_col: "living_pop_daily_avg"})
    records.append(df_y)
    print(f"  {year}: {df_y.shape}")

df_lp = pd.concat(records, ignore_index=True)
df_lp.to_csv(OUT / "living_pop_daily_avg_by_gu_year.csv", index=False, encoding="utf-8-sig")
print("\n저장 완료:", df_lp.shape)
df_lp.head(3)

## 2. 장기체류 외국인 생활인구

In [ ]:
records_f = []
for year in range(2020, 2025):
    zip_path = RAW / f"LONG_FOREIGNER_GU_{year}.zip"
    if not zip_path.exists():
        continue
    df = read_lp_zip(zip_path)
    pop_col = "총생활인구수"
    df_y = aggregate_to_annual(df, pop_col)
    df_y = df_y.rename(columns={pop_col: "foreigner_lp_daily_avg"})
    records_f.append(df_y)
    print(f"  외국인 {year}: {df_y.shape}")

df_f = pd.concat(records_f, ignore_index=True)
df_f.to_csv(OUT / "foreigner_lp_daily_avg_by_gu_year.csv", index=False, encoding="utf-8-sig")
print("\n저장 완료:", df_f.shape)

## 3. 주간/야간 생활인구 비율 (유동인구형 자치구 판별용)

In [ ]:
# 주간(07~21시): 시간대구분 7~20, 야간(22~06시): 나머지
# 시간대구분 컬럼: 0~23 (0=0~1시, 1=1~2시, ... 23=23~24시)
records_dn = []
for year in range(2020, 2025):
    zip_path = RAW / f"LOCAL_PEOPLE_GU_{year}.zip"
    if not zip_path.exists():
        continue
    df = read_lp_zip(zip_path)
    df["daytime"] = df["시간대구분"].between(7, 20)  # 07~20시
    df_dn = (
        df.groupby(["year","month","date","gu","daytime"])["총생활인구수"]
        .sum().reset_index()
    )
    df_day_avg = (
        df_dn.groupby(["year","gu","daytime"])["총생활인구수"]
        .mean().reset_index()
    )
    df_pivot = df_day_avg.pivot(index=["year","gu"], columns="daytime", values="총생활인구수").reset_index()
    df_pivot.columns = ["year","gu","nighttime_avg","daytime_avg"]
    df_pivot["day_night_ratio"] = df_pivot["daytime_avg"] / df_pivot["nighttime_avg"]
    records_dn.append(df_pivot)
    print(f"  {year}: {df_pivot.shape}")

df_dn_all = pd.concat(records_dn, ignore_index=True)
df_dn_all.to_csv(OUT / "day_night_ratio_by_gu_year.csv", index=False, encoding="utf-8-sig")
print("\n저장 완료:", df_dn_all.shape)
df_dn_all.sort_values("day_night_ratio", ascending=False).head(5)

## 4. 병합 — 생활인구 통합 테이블

In [ ]:
df_merged = df_lp.merge(df_f, on=["year","gu"], how="left")
df_merged = df_merged.merge(
    df_dn_all[["year","gu","daytime_avg","nighttime_avg","day_night_ratio"]],
    on=["year","gu"], how="left"
)
df_merged.to_csv(OUT / "living_pop_merged_by_gu_year.csv", index=False, encoding="utf-8-sig")
print("병합 저장:", df_merged.shape)
df_merged.head(3)

## 5. 최종 검증

In [ ]:
for f in ["living_pop_daily_avg_by_gu_year.csv","foreigner_lp_daily_avg_by_gu_year.csv",
          "day_night_ratio_by_gu_year.csv","living_pop_merged_by_gu_year.csv"]:
    df = pd.read_csv(OUT / f)
    print(f"  {f}: {df.shape}, gu={df['gu'].nunique()}, years={sorted(df['year'].unique())}")